# 10 Spielstil, Teamstaerke und Wetter-Intensitaet analysieren

Dieses Notebook implementiert BD-20 und BD-21. Es vergleicht zentrale Spielstil- und Intensitaetsmetriken nach Turnier, Elo-Gruppe und Wetterbedingungen.

Input:
- `data/gold/team_match_analysis_dataset.csv`

Outputs:
- BD-20-Grafiken zu xG, Passquote, angekommenen Paessen und Pressures nach Turnier/Elo-Gruppe
- BD-21-Grafiken zu Temperatur, gefuehlter Temperatur, Regen und Intensitaet
- Summary-Tabellen in `outputs/tables/`


## Methodischer Ansatz

Das finale Gold-Dataset enthaelt eine Zeile pro Team und Spiel. Fuer Spielstil- und Intensitaetsmetriken ist diese Team-Match-Ebene passend, weil xG, Passquote, angekommene Paesse, Pressures, Counterpressures und Duels aus Sicht eines Teams interpretiert werden.

Der Aufbau folgt den Kursressourcen zu Data Analysis und Spark/DataFrames: zuerst werden Daten tabellarisch gelesen und validiert, danach werden Spalten mit `groupby` aggregiert und die Ergebnisse als reproduzierbare Artefakte gespeichert. Fuer Wettervergleiche bleibt die Analyse ebenfalls auf Team-Match-Ebene, weil die Frage lautet, ob Teams unter bestimmten Wetterbedingungen anders intensiv spielen.

Die Analysen in diesem Notebook sind explorativ. Multivariate Kontrollanalysen, zum Beispiel mit Elo-Differenz, Turnier und Wettervariablen im selben Regressionsmodell, werden separat in einem Modell-Notebook behandelt.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from project_paths import project_root

ROOT = project_root()
DATA_PATH = ROOT / 'data' / 'gold' / 'team_match_analysis_dataset.csv'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RAIN_THRESHOLD_MM = 0.5
RAIN_GROUP_ORDER = ['kein/kaum Regen', 'messbarer Regen']

XG_BY_TOURNAMENT_PATH = FIGURE_DIR / 'xg_by_tournament.png'
PASS_BY_TOURNAMENT_PATH = FIGURE_DIR / 'pass_completion_by_tournament.png'
SUCCESSFUL_PASSES_BY_TOURNAMENT_PATH = FIGURE_DIR / 'successful_passes_by_tournament.png'
PRESSURES_BY_TOURNAMENT_PATH = FIGURE_DIR / 'pressures_by_tournament.png'
XG_BY_ELO_GROUP_PATH = FIGURE_DIR / 'xg_by_elo_group.png'
PASS_BY_ELO_GROUP_PATH = FIGURE_DIR / 'pass_completion_by_elo_group.png'
SUCCESSFUL_PASSES_BY_ELO_GROUP_PATH = FIGURE_DIR / 'successful_passes_by_elo_group.png'
TEMPERATURE_VS_PRESSURES_PATH = FIGURE_DIR / 'temperature_vs_pressures.png'
FEELS_LIKE_VS_PRESSURES_PATH = FIGURE_DIR / 'feels_like_vs_pressures.png'
PRESSURES_BY_TEMPERATURE_GROUP_PATH = FIGURE_DIR / 'pressures_by_temperature_group.png'
PRESSURES_RAIN_VS_NO_RAIN_PATH = FIGURE_DIR / 'pressures_rain_vs_no_rain.png'
DUELS_BY_TEMPERATURE_GROUP_PATH = FIGURE_DIR / 'duels_by_temperature_group.png'
TEMPERATURE_VS_PASS_COMPLETION_PATH = FIGURE_DIR / 'temperature_vs_pass_completion.png'
FEELS_LIKE_VS_PASS_COMPLETION_PATH = FIGURE_DIR / 'feels_like_vs_pass_completion.png'
PASS_COMPLETION_BY_TEMPERATURE_GROUP_PATH = FIGURE_DIR / 'pass_completion_by_temperature_group.png'
PASS_COMPLETION_RAIN_VS_NO_RAIN_PATH = FIGURE_DIR / 'pass_completion_rain_vs_no_rain.png'
LONG_BALL_SHARE_RAIN_VS_NO_RAIN_PATH = FIGURE_DIR / 'long_ball_share_rain_vs_no_rain.png'
SUMMARY_PATH = TABLE_DIR / 'bd20_style_intensity_summary.csv'
WEATHER_INTENSITY_SUMMARY_PATH = TABLE_DIR / 'bd21_weather_intensity_summary.csv'
BALL_CONTROL_SUMMARY_PATH = TABLE_DIR / 'bd22_weather_ball_control_summary.csv'

required_columns = [
    'match_id',
    'team_id',
    'short_name',
    'competition_name',
    'team_name',
    'xg',
    'passes',
    'successful_passes',
    'pass_completion_rate',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
    'long_ball_share',
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'temperature_group',
    'elo_group',
]

metric_colors = {
    'xg': '#3569a8',
    'pass_completion_rate': '#2f7d58',
    'successful_passes': '#5c7c2f',
    'pressures': '#9a5b2f',
    'weather': '#4f6f52',
    'rain': '#456990',
    'duels': '#8a4f7d',
    'ball_control': '#2f7d76',
    'long_ball': '#7a5c2e',
}


## Daten laden

Die Analyse liest nur aus dem Gold-Dataset. Die Checks stellen sicher, dass BD20 nicht auf unvollstaendigen Inputs laeuft.

In [ ]:
assert DATA_PATH.exists(), f'Missing required input file: {DATA_PATH}'

team_match_df = pd.read_csv(DATA_PATH)
missing_columns = [column for column in required_columns if column not in team_match_df.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

analysis_df = team_match_df[required_columns].copy()
analysis_df['pass_completion_pct'] = analysis_df['pass_completion_rate'] * 100
analysis_df['long_ball_share_pct'] = analysis_df['long_ball_share'] * 100
analysis_df['rain_group'] = np.where(
    analysis_df['rain_mm'] > RAIN_THRESHOLD_MM,
    RAIN_GROUP_ORDER[1],
    RAIN_GROUP_ORDER[0],
)
analysis_df['rain_group'] = pd.Categorical(
    analysis_df['rain_group'],
    categories=RAIN_GROUP_ORDER,
    ordered=True,
)

required_metrics = [
    'xg',
    'passes',
    'successful_passes',
    'pass_completion_pct',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
    'long_ball_share_pct',
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'temperature_group',
]
missing_metric_values = analysis_df[required_metrics + ['short_name', 'elo_group']].isna().sum()
assert missing_metric_values.sum() == 0, f'Unexpected missing values: {missing_metric_values.to_dict()}'
assert not analysis_df.duplicated(['match_id', 'team_id']).any(), 'Expected one row per match/team pair.'
assert analysis_df['temperature_group'].nunique() >= 2, 'Expected at least two temperature groups for comparison.'
assert analysis_df['rain_group'].nunique() == 2, 'Expected both rain groups for comparison.'

analysis_df.head()


## Gruppierte Kennzahlen

Die Tabelle fasst Mittelwerte und Mediane nach Turnier und Elo-Gruppe zusammen. Sie ist der numerische Gegenpart zu den Visualisierungen.

In [ ]:
def summarize_group(dataframe, group_column, group_type):
    summary = (
        dataframe.groupby(group_column, sort=False)
        .agg(
            team_matches=('match_id', 'size'),
            xg_mean=('xg', 'mean'),
            xg_median=('xg', 'median'),
            passes_mean=('passes', 'mean'),
            successful_passes_mean=('successful_passes', 'mean'),
            successful_passes_median=('successful_passes', 'median'),
            pass_completion_mean=('pass_completion_pct', 'mean'),
            pass_completion_median=('pass_completion_pct', 'median'),
            carries_mean=('carries', 'mean'),
            carries_median=('carries', 'median'),
            long_balls_mean=('long_balls', 'mean'),
            long_ball_share_mean=('long_ball_share_pct', 'mean'),
            long_ball_share_median=('long_ball_share_pct', 'median'),
            pressures_mean=('pressures', 'mean'),
            pressures_median=('pressures', 'median'),
            counterpressures_mean=('counterpressures', 'mean'),
            duels_mean=('duels', 'mean'),
        )
        .reset_index()
        .rename(columns={group_column: 'group'})
    )
    summary.insert(0, 'group_type', group_type)
    return summary


tournament_summary = summarize_group(analysis_df, 'short_name', 'tournament')
elo_order = ['underdog', 'balanced', 'favorite']
elo_summary = summarize_group(
    analysis_df.assign(elo_group=pd.Categorical(analysis_df['elo_group'], categories=elo_order, ordered=True))
    .sort_values('elo_group'),
    'elo_group',
    'elo_group',
)

style_summary = pd.concat([tournament_summary, elo_summary], ignore_index=True)
numeric_columns = style_summary.select_dtypes(include='number').columns
style_summary[numeric_columns] = style_summary[numeric_columns].round(3)
style_summary.to_csv(SUMMARY_PATH, index=False)
style_summary

## Plot-Helfer

Die Visualisierungen verwenden denselben Aufbau, damit Turniere und Elo-Gruppen direkt vergleichbar bleiben. Der Balken zeigt den Mittelwert; die kurze dunkle Linie im Balken zeigt den Median.

In [ ]:
def grouped_metric(dataframe, group_column, value_column, order=None):
    result = (
        dataframe.groupby(group_column, sort=False)[value_column]
        .agg(['mean', 'median', 'count'])
        .reset_index()
    )
    if order is None:
        return result

    result[group_column] = pd.Categorical(result[group_column], categories=order, ordered=True)
    return result.sort_values(group_column).reset_index(drop=True)


def save_bar_chart(grouped, group_column, title, ylabel, output_path, color):
    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=160)
    x_positions = range(len(grouped))
    labels = grouped[group_column].astype(str)
    bars = ax.bar(x_positions, grouped['mean'], color=color)

    for x_position, median in zip(x_positions, grouped['median']):
        ax.hlines(
            y=median,
            xmin=x_position - 0.22,
            xmax=x_position + 0.22,
            color='#222222',
            linewidth=2,
            zorder=3,
        )

    ax.set_title(title, loc='left', fontweight='bold', pad=24)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.set_xticks(list(x_positions), labels)
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='x', rotation=20)

    value_range = grouped['mean'].max() - grouped['mean'].min()
    label_offset = max(value_range * 0.04, grouped['mean'].max() * 0.015, 0.05)
    for bar, count in zip(bars, grouped['count']):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + label_offset,
            f'{height:.1f}\nn={int(count)}',
            ha='center',
            va='bottom',
            fontsize=9,
        )

    ax.set_ylim(0, grouped['mean'].max() + label_offset * 5)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()



## xG nach Turnier

xG beschreibt, wie viele erwartete Tore ein Team pro Spiel erzeugt. Unterschiede koennen auf Turnierkontext, Gegnerniveau oder Spielanlage hindeuten. Weil xG rechtsschief verteilt ist, koennen einzelne sehr chancenreiche Spiele den Mittelwert sichtbar ueber den Median ziehen.

In [ ]:
xg_by_tournament = grouped_metric(analysis_df, 'short_name', 'xg')
save_bar_chart(
    xg_by_tournament,
    group_column='short_name',
    title='xG nach Turnier',
    ylabel='Durchschnittliches xG pro Team-Spiel',
    output_path=XG_BY_TOURNAMENT_PATH,
    color=metric_colors['xg'],
)
xg_by_tournament

## Passquote nach Turnier

Die Passquote zeigt, wie sauber Teams im jeweiligen Turnier den Ballbesitz ausspielen konnten.

In [ ]:
pass_completion_by_tournament = grouped_metric(analysis_df, 'short_name', 'pass_completion_pct')
save_bar_chart(
    pass_completion_by_tournament,
    group_column='short_name',
    title='Passquote nach Turnier',
    ylabel='Durchschnittliche Passquote in %',
    output_path=PASS_BY_TOURNAMENT_PATH,
    color=metric_colors['pass_completion_rate'],
)
pass_completion_by_tournament

## Angekommene Paesse nach Turnier

Die Zahl der angekommenen Paesse ergaenzt die Passquote um Ballkontroll-Volumen. Eine hohe Passquote allein kann aus wenigen sicheren Paessen entstehen; viele erfolgreiche Paesse sprechen staerker fuer strukturelle Kontrolle im Ballbesitz.


In [ ]:
successful_passes_by_tournament = grouped_metric(analysis_df, 'short_name', 'successful_passes')
save_bar_chart(
    successful_passes_by_tournament,
    group_column='short_name',
    title='Angekommene Paesse nach Turnier',
    ylabel='Durchschnittliche angekommene Paesse pro Team-Spiel',
    output_path=SUCCESSFUL_PASSES_BY_TOURNAMENT_PATH,
    color=metric_colors['successful_passes'],
)
successful_passes_by_tournament


## Pressures nach Turnier

Pressures dienen als Intensitaetsindikator. Hoehere Werte sprechen fuer aktiveres Stoeren des Gegners.

In [ ]:
pressures_by_tournament = grouped_metric(analysis_df, 'short_name', 'pressures')
save_bar_chart(
    pressures_by_tournament,
    group_column='short_name',
    title='Pressures nach Turnier',
    ylabel='Durchschnittliche Pressures pro Team-Spiel',
    output_path=PRESSURES_BY_TOURNAMENT_PATH,
    color=metric_colors['pressures'],
)
pressures_by_tournament

## xG nach Elo-Gruppe

Die Elo-Gruppe vergleicht Favoriten, ausgeglichene Matchups und Aussenseiter aus Sicht des jeweiligen Teams. Auch hier sollte der Mittelwert zusammen mit der Medianlinie gelesen werden, weil einzelne Spiele mit sehr hohem xG den Durchschnitt beeinflussen koennen.

In [ ]:
xg_by_elo_group = grouped_metric(analysis_df, 'elo_group', 'xg', order=elo_order)
save_bar_chart(
    xg_by_elo_group,
    group_column='elo_group',
    title='xG nach Elo-Gruppe',
    ylabel='Durchschnittliches xG pro Team-Spiel',
    output_path=XG_BY_ELO_GROUP_PATH,
    color=metric_colors['xg'],
)
xg_by_elo_group

## Passquote nach Elo-Gruppe

Die Passquote nach Elo-Gruppe zeigt, ob staerkere Teams den Ballbesitz strukturell sauberer halten.

In [ ]:
pass_completion_by_elo_group = grouped_metric(analysis_df, 'elo_group', 'pass_completion_pct', order=elo_order)
save_bar_chart(
    pass_completion_by_elo_group,
    group_column='elo_group',
    title='Passquote nach Elo-Gruppe',
    ylabel='Durchschnittliche Passquote in %',
    output_path=PASS_BY_ELO_GROUP_PATH,
    color=metric_colors['pass_completion_rate'],
)
pass_completion_by_elo_group

## Angekommene Paesse nach Elo-Gruppe

Die Elo-Gruppen zeigen, ob Favoriten nicht nur praeziser passen, sondern auch mehr erfolgreiche Paesse sammeln. Das trennt Passsicherheit von Ballkontroll-Volumen.


In [ ]:
successful_passes_by_elo_group = grouped_metric(analysis_df, 'elo_group', 'successful_passes', order=elo_order)
save_bar_chart(
    successful_passes_by_elo_group,
    group_column='elo_group',
    title='Angekommene Paesse nach Elo-Gruppe',
    ylabel='Durchschnittliche angekommene Paesse pro Team-Spiel',
    output_path=SUCCESSFUL_PASSES_BY_ELO_GROUP_PATH,
    color=metric_colors['successful_passes'],
)
successful_passes_by_elo_group


## BD-21: Wetter und Intensitaet

Die folgenden Grafiken pruefen, ob Intensitaetsmetriken mit Wetterbedingungen vergleichbar sind. Pressures stehen im Mittelpunkt, weil sie im Issue als zentrale Metrik genannt sind. Fuer Regen wird nicht jedes minimale `rain_mm > 0` als eigene Regenbedingung gelesen: Werte bis 0.5 mm werden als `kein/kaum Regen` zusammengefasst, weil sehr leichter Niederschlag sportlich schwer interpretierbar ist.


In [ ]:
temperature_order = ['cold', 'mild', 'warm', 'hot']
observed_temperature_order = [group for group in temperature_order if group in set(analysis_df['temperature_group'])]
ordered_temperature_df = analysis_df.assign(
    temperature_group=pd.Categorical(
        analysis_df['temperature_group'],
        categories=observed_temperature_order,
        ordered=True,
    )
).sort_values('temperature_group')

weather_intensity_summary = pd.concat(
    [
        summarize_group(ordered_temperature_df, 'temperature_group', 'temperature_group'),
        summarize_group(analysis_df.sort_values('rain_group'), 'rain_group', 'rain_group'),
    ],
    ignore_index=True,
)
numeric_columns = weather_intensity_summary.select_dtypes(include='number').columns
weather_intensity_summary[numeric_columns] = weather_intensity_summary[numeric_columns].round(3)
weather_intensity_summary.to_csv(WEATHER_INTENSITY_SUMMARY_PATH, index=False)
weather_intensity_summary


## Plot-Helfer fuer Wettervergleiche

Scatterplots zeigen die kontinuierliche Beziehung zwischen Temperatur und Spielmetriken. Boxplots verdichten Gruppenvergleiche fuer Temperaturgruppen und Regenbedingungen.


In [ ]:
def save_scatter_with_trend(dataframe, x_column, y_column, title, xlabel, ylabel, output_path, color):
    plot_df = dataframe[[x_column, y_column]].dropna()
    correlation = plot_df[x_column].corr(plot_df[y_column])

    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=160)
    ax.scatter(plot_df[x_column], plot_df[y_column], alpha=0.65, s=36, color=color, edgecolor='white', linewidth=0.4)

    if plot_df[x_column].nunique() > 1:
        slope, intercept = np.polyfit(plot_df[x_column], plot_df[y_column], 1)
        x_values = np.linspace(plot_df[x_column].min(), plot_df[x_column].max(), 100)
        ax.plot(x_values, slope * x_values + intercept, color='#222222', linewidth=2)

    ax.set_title(title, loc='left', fontweight='bold', pad=24)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.text(
        0.02,
        0.96,
        f'n={len(plot_df)} | r={correlation:.2f}',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=10,
        bbox={'facecolor': 'white', 'edgecolor': '#dddddd', 'boxstyle': 'round,pad=0.35'},
    )
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()
    return {'n': int(len(plot_df)), 'correlation': round(float(correlation), 3)}


def save_boxplot(dataframe, group_column, value_column, order, title, xlabel, ylabel, output_path, color):
    available_order = [group for group in order if group in set(dataframe[group_column].dropna())]
    values = [dataframe.loc[dataframe[group_column] == group, value_column].dropna() for group in available_order]
    counts = [len(series) for series in values]

    assert available_order, f'No groups available for {group_column}'
    assert all(count > 0 for count in counts), f'Empty group found for {group_column}'

    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=160)
    box = ax.boxplot(values, labels=available_order, patch_artist=True, showmeans=True)

    for patch in box['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.72)
    for median in box['medians']:
        median.set_color('#222222')
        median.set_linewidth(2)
    for mean in box['means']:
        mean.set_marker('o')
        mean.set_markerfacecolor('white')
        mean.set_markeredgecolor('#222222')

    y_top = max(series.max() for series in values)
    y_bottom = min(series.min() for series in values)
    label_offset = max((y_top - y_bottom) * 0.04, y_top * 0.02, 1)
    for index, count in enumerate(counts, start=1):
        ax.text(index, y_top + label_offset, f'n={count}', ha='center', va='bottom', fontsize=9)

    ax.set_title(title, loc='left', fontweight='bold', pad=24)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(bottom=max(0, y_bottom - label_offset), top=y_top + label_offset * 4)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()

    return pd.DataFrame({
        group_column: available_order,
        'count': counts,
        'mean': [round(float(series.mean()), 2) for series in values],
        'median': [round(float(series.median()), 2) for series in values],
    })


## Temperatur vs. Pressures

Der Scatterplot vergleicht gemessene Temperatur und Pressures pro Team-Spiel. Die Trendlinie ist bewusst einfach gehalten und dient nur als Orientierung, nicht als kausales Modell.

In [ ]:
temperature_pressure_relation = save_scatter_with_trend(
    analysis_df,
    x_column='temperature_c',
    y_column='pressures',
    title='Temperatur vs. Pressures',
    xlabel='Temperatur bei Anpfiff in Grad Celsius',
    ylabel='Pressures pro Team-Spiel',
    output_path=TEMPERATURE_VS_PRESSURES_PATH,
    color=metric_colors['weather'],
)
temperature_pressure_relation

## Gefuehlte Temperatur vs. Pressures

Die gefuehlte Temperatur beruecksichtigt zusaetzliche Wettereffekte. Sie kann deshalb einen leicht anderen Zusammenhang mit Intensitaet zeigen als die reine Lufttemperatur.

In [ ]:
feels_like_pressure_relation = save_scatter_with_trend(
    analysis_df,
    x_column='feels_like_c',
    y_column='pressures',
    title='Gefuehlte Temperatur vs. Pressures',
    xlabel='Gefuehlte Temperatur bei Anpfiff in Grad Celsius',
    ylabel='Pressures pro Team-Spiel',
    output_path=FEELS_LIKE_VS_PRESSURES_PATH,
    color=metric_colors['weather'],
)
feels_like_pressure_relation

## Pressures nach Temperaturgruppe

Die Temperaturgruppen machen die Streuung leichter vergleichbar. Der weisse Punkt markiert den Mittelwert, die dunkle Linie den Median.

In [ ]:
pressures_by_temperature_group = save_boxplot(
    analysis_df,
    group_column='temperature_group',
    value_column='pressures',
    order=observed_temperature_order,
    title='Pressures nach Temperaturgruppe',
    xlabel='Temperaturgruppe',
    ylabel='Pressures pro Team-Spiel',
    output_path=PRESSURES_BY_TEMPERATURE_GROUP_PATH,
    color=metric_colors['weather'],
)
pressures_by_temperature_group

## Pressures nach Regen-Gruppe

Der Boxplot vergleicht Team-Spiele mit `kein/kaum Regen` (bis 0.5 mm) und `messbarer Regen` (mehr als 0.5 mm). Die zweite Gruppe ist deutlich kleiner und sollte deshalb vorsichtig interpretiert werden.


In [ ]:
pressures_by_rain_group = save_boxplot(
    analysis_df,
    group_column='rain_group',
    value_column='pressures',
    order=RAIN_GROUP_ORDER,
    title='Pressures nach Regen-Gruppe',
    xlabel='Regenbedingung',
    ylabel='Pressures pro Team-Spiel',
    output_path=PRESSURES_RAIN_VS_NO_RAIN_PATH,
    color=metric_colors['rain'],
)
pressures_by_rain_group


## Optional: Duels nach Temperaturgruppe

Duels ergaenzen Pressures als zweite koerperliche Intensitaetsmetrik. Der Plot zeigt, ob Zweikampfvolumen in den Temperaturgruppen anders verteilt ist.

In [ ]:
duels_by_temperature_group = save_boxplot(
    analysis_df,
    group_column='temperature_group',
    value_column='duels',
    order=observed_temperature_order,
    title='Duels nach Temperaturgruppe',
    xlabel='Temperaturgruppe',
    ylabel='Duels pro Team-Spiel',
    output_path=DUELS_BY_TEMPERATURE_GROUP_PATH,
    color=metric_colors['duels'],
)
duels_by_temperature_group

## BD-22: Wetter und Ballkontrolle

Die Hauptanalyse nutzt kontinuierliche Wetterwerte, weil Temperatur- und Regen-Grenzen sonst fachlich zu hart wirken koennen. Die Successful-Pass-Quote ist hier `successful_passes / passes` und entspricht `pass_completion_pct`. Ergaenzend zeigen Paesse, Carries und Long-Ball-Share, ob sich nicht nur die Praezision, sondern auch Ballkontroll-Volumen und Spielanlage veraendern.

In [ ]:
ball_control_summary = pd.concat(
    [
        summarize_group(ordered_temperature_df, 'temperature_group', 'temperature_group'),
        summarize_group(analysis_df.sort_values('rain_group'), 'rain_group', 'rain_group'),
    ],
    ignore_index=True,
)
numeric_columns = ball_control_summary.select_dtypes(include='number').columns
ball_control_summary[numeric_columns] = ball_control_summary[numeric_columns].round(3)
ball_control_summary.to_csv(BALL_CONTROL_SUMMARY_PATH, index=False)
ball_control_summary


## Temperatur vs. Successful-Pass-Quote

Der Scatterplot zeigt die Successful-Pass-Quote pro Team-Spiel gegen die gemessene Temperatur bei Anpfiff. Diese Sicht ist fuer Temperatur die staerkste Option, weil keine Information durch willkuerliche Temperaturgruppen verloren geht.

In [ ]:
temperature_pass_relation = save_scatter_with_trend(
    analysis_df,
    x_column='temperature_c',
    y_column='pass_completion_pct',
    title='Temperatur vs. Successful-Pass-Quote',
    xlabel='Temperatur bei Anpfiff in Grad Celsius',
    ylabel='Successful-Pass-Quote in Prozent',
    output_path=TEMPERATURE_VS_PASS_COMPLETION_PATH,
    color=metric_colors['ball_control'],
)
temperature_pass_relation

## Gefuehlte Temperatur vs. Successful-Pass-Quote

Die gefuehlte Temperatur kombiniert die Lufttemperatur mit weiteren Wettereffekten. Der Vergleich mit der Successful-Pass-Quote zeigt, ob die wahrgenommene Belastung staerker mit Passpraezision verbunden ist als die reine Temperatur.

In [ ]:
feels_like_pass_relation = save_scatter_with_trend(
    analysis_df,
    x_column='feels_like_c',
    y_column='pass_completion_pct',
    title='Gefuehlte Temperatur vs. Successful-Pass-Quote',
    xlabel='Gefuehlte Temperatur bei Anpfiff in Grad Celsius',
    ylabel='Successful-Pass-Quote in Prozent',
    output_path=FEELS_LIKE_VS_PASS_COMPLETION_PATH,
    color=metric_colors['ball_control'],
)
feels_like_pass_relation

## Successful-Pass-Quote nach Temperaturgruppe

Der Boxplot nach Temperaturgruppe ist nur eine verdichtete Zusatzsicht zum Scatterplot. Er eignet sich gut fuer eine Folie, sollte aber nicht allein als Beleg gelesen werden, weil die Gruppen ungleich gross sind und keine kalte Gruppe beobachtet wurde.

In [ ]:
pass_completion_by_temperature_group = save_boxplot(
    analysis_df,
    group_column='temperature_group',
    value_column='pass_completion_pct',
    order=observed_temperature_order,
    title='Successful-Pass-Quote nach Temperaturgruppe',
    xlabel='Temperaturgruppe',
    ylabel='Successful-Pass-Quote in Prozent',
    output_path=PASS_COMPLETION_BY_TEMPERATURE_GROUP_PATH,
    color=metric_colors['ball_control'],
)
pass_completion_by_temperature_group

## Successful-Pass-Quote nach Regen-Gruppe

Der Boxplot ersetzt den kontinuierlichen Regen-Scatterplot, weil sehr viele Team-Spiele 0 mm Niederschlag haben. Die Gruppierung nutzt dieselbe 0.5-mm-Schwelle wie die Intensitaetsanalyse.


In [ ]:
pass_completion_by_rain_group = save_boxplot(
    analysis_df,
    group_column='rain_group',
    value_column='pass_completion_pct',
    order=RAIN_GROUP_ORDER,
    title='Successful-Pass-Quote nach Regen-Gruppe',
    xlabel='Regenbedingung',
    ylabel='Successful-Pass-Quote in Prozent',
    output_path=PASS_COMPLETION_RAIN_VS_NO_RAIN_PATH,
    color=metric_colors['rain'],
)
pass_completion_by_rain_group


## Long-Ball-Share nach Regen-Gruppe

Long-Ball-Share ergaenzt die Passquote um Spielanlage. Auch hier ist der Vergleich explorativ, weil die Gruppe mit messbarem Regen klein bleibt.


In [ ]:
long_ball_share_by_rain_group = save_boxplot(
    analysis_df,
    group_column='rain_group',
    value_column='long_ball_share_pct',
    order=RAIN_GROUP_ORDER,
    title='Long-Ball-Share nach Regen-Gruppe',
    xlabel='Regenbedingung',
    ylabel='Long-Ball-Share in Prozent',
    output_path=LONG_BALL_SHARE_RAIN_VS_NO_RAIN_PATH,
    color=metric_colors['long_ball'],
)
long_ball_share_by_rain_group


## Output-Checks

Die Checks stellen sicher, dass alle geforderten BD-20- und BD-21-Artefakte geschrieben wurden und nicht leer sind.

In [ ]:
figure_paths = [
    XG_BY_TOURNAMENT_PATH,
    PASS_BY_TOURNAMENT_PATH,
    SUCCESSFUL_PASSES_BY_TOURNAMENT_PATH,
    PRESSURES_BY_TOURNAMENT_PATH,
    XG_BY_ELO_GROUP_PATH,
    PASS_BY_ELO_GROUP_PATH,
    SUCCESSFUL_PASSES_BY_ELO_GROUP_PATH,
    TEMPERATURE_VS_PRESSURES_PATH,
    FEELS_LIKE_VS_PRESSURES_PATH,
    PRESSURES_BY_TEMPERATURE_GROUP_PATH,
    PRESSURES_RAIN_VS_NO_RAIN_PATH,
    DUELS_BY_TEMPERATURE_GROUP_PATH,
    TEMPERATURE_VS_PASS_COMPLETION_PATH,
    FEELS_LIKE_VS_PASS_COMPLETION_PATH,
    PASS_COMPLETION_BY_TEMPERATURE_GROUP_PATH,
    PASS_COMPLETION_RAIN_VS_NO_RAIN_PATH,
    LONG_BALL_SHARE_RAIN_VS_NO_RAIN_PATH,
]

table_paths = [SUMMARY_PATH, WEATHER_INTENSITY_SUMMARY_PATH, BALL_CONTROL_SUMMARY_PATH]

for path in figure_paths:
    assert path.exists(), f'Missing expected figure: {path}'
    assert path.stat().st_size > 1_000, f'Figure looks too small or empty: {path}'

for path in table_paths:
    assert path.exists(), f'Missing expected summary table: {path}'
    assert path.stat().st_size > 200, f'Summary table looks too small or empty: {path}'

pd.DataFrame({
    'artifact': [str(path.relative_to(ROOT)) for path in figure_paths + table_paths],
    'bytes': [path.stat().st_size for path in figure_paths + table_paths],
})


## Interpretation fuer BD-20, BD-21 und BD-22

Die Turniere unterscheiden sich sichtbar in xG, Successful-Pass-Quote, angekommenen Paessen und Pressingintensitaet. Angekommene Paesse werden als Ballkontroll-Volumen genutzt, weil sie die Passquote um Spielanteile ergaenzen. Fuer Regen ist die Interpretation bewusst defensiv: Bis 0.5 mm wird als kein/kaum Regen gelesen; die Gruppe mit messbarem Regen bleibt klein und eignet sich eher fuer vorsichtige Vergleiche als fuer starke Schluesse.


In [ ]:
interpretation = {
    'team_matches': int(len(analysis_df)),
    'tournaments': int(analysis_df['short_name'].nunique()),
    'rain_threshold_mm': RAIN_THRESHOLD_MM,
    'low_or_no_rain_team_matches': int((analysis_df['rain_mm'] <= RAIN_THRESHOLD_MM).sum()),
    'measurable_rain_team_matches': int((analysis_df['rain_mm'] > RAIN_THRESHOLD_MM).sum()),
    'highest_xg_tournament': xg_by_tournament.sort_values('mean', ascending=False).iloc[0]['short_name'],
    'highest_pass_completion_tournament': pass_completion_by_tournament.sort_values('mean', ascending=False).iloc[0]['short_name'],
    'highest_successful_passes_tournament': successful_passes_by_tournament.sort_values('mean', ascending=False).iloc[0]['short_name'],
    'highest_pressure_tournament': pressures_by_tournament.sort_values('mean', ascending=False).iloc[0]['short_name'],
    'favorite_xg_mean': round(float(xg_by_elo_group.loc[xg_by_elo_group['elo_group'] == 'favorite', 'mean'].iloc[0]), 2),
    'underdog_xg_mean': round(float(xg_by_elo_group.loc[xg_by_elo_group['elo_group'] == 'underdog', 'mean'].iloc[0]), 2),
    'favorite_pass_completion_pct': round(float(pass_completion_by_elo_group.loc[pass_completion_by_elo_group['elo_group'] == 'favorite', 'mean'].iloc[0]), 1),
    'underdog_pass_completion_pct': round(float(pass_completion_by_elo_group.loc[pass_completion_by_elo_group['elo_group'] == 'underdog', 'mean'].iloc[0]), 1),
    'favorite_successful_passes_mean': round(float(successful_passes_by_elo_group.loc[successful_passes_by_elo_group['elo_group'] == 'favorite', 'mean'].iloc[0]), 1),
    'underdog_successful_passes_mean': round(float(successful_passes_by_elo_group.loc[successful_passes_by_elo_group['elo_group'] == 'underdog', 'mean'].iloc[0]), 1),
    'temperature_pressure_correlation': temperature_pressure_relation['correlation'],
    'feels_like_pressure_correlation': feels_like_pressure_relation['correlation'],
    'highest_pressure_temperature_group': pressures_by_temperature_group.sort_values('mean', ascending=False).iloc[0]['temperature_group'],
    'low_or_no_rain_pressure_mean': float(pressures_by_rain_group.loc[pressures_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[0], 'mean'].iloc[0]),
    'measurable_rain_pressure_mean': float(pressures_by_rain_group.loc[pressures_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[1], 'mean'].iloc[0]),
    'highest_duels_temperature_group': duels_by_temperature_group.sort_values('mean', ascending=False).iloc[0]['temperature_group'],
    'temperature_pass_completion_correlation': temperature_pass_relation['correlation'],
    'feels_like_pass_completion_correlation': feels_like_pass_relation['correlation'],
    'highest_pass_completion_temperature_group': pass_completion_by_temperature_group.sort_values('mean', ascending=False).iloc[0]['temperature_group'],
    'low_or_no_rain_pass_completion_pct': float(pass_completion_by_rain_group.loc[pass_completion_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[0], 'mean'].iloc[0]),
    'measurable_rain_pass_completion_pct': float(pass_completion_by_rain_group.loc[pass_completion_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[1], 'mean'].iloc[0]),
    'low_or_no_rain_long_ball_share_pct': float(long_ball_share_by_rain_group.loc[long_ball_share_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[0], 'mean'].iloc[0]),
    'measurable_rain_long_ball_share_pct': float(long_ball_share_by_rain_group.loc[long_ball_share_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[1], 'mean'].iloc[0]),
}
interpretation
